In [5]:
import requests
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

plt.style.use('default')

SYMBOL = "BTCUSDT"
ORDER_BOOK_LIMIT = 10
ITERATIONS = 50
SLEEP_TIME = 3

WINDOW_SIZE = 20
Z_THRESHOLD = 2

def get_order_book(symbol):
    url = f"https://api.binance.com/api/v3/depth?symbol={symbol}&limit={ORDER_BOOK_LIMIT}"
    response = requests.get(url)
    
    if response.status_code != 200:
        raise Exception("API Error")
    
    data = response.json()
    
    bids = pd.DataFrame(data['bids'], columns=['price', 'quantity'], dtype=float)
    asks = pd.DataFrame(data['asks'], columns=['price', 'quantity'], dtype=float)
    
    return bids, asks


def get_price(symbol):
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    
    if response.status_code != 200:
        raise Exception("API Error")
    
    data = response.json()
    return float(data['price'])

def compute_metrics(bids, asks):
    best_bid = bids['price'].max()
    best_ask = asks['price'].min()
    
    mid_price = (best_bid + best_ask) / 2
    spread = best_ask - best_bid
    
    bid_depth = bids['quantity'].sum()
    ask_depth = asks['quantity'].sum()
    
    return mid_price, spread, bid_depth, ask_depth

results = []

for i in range(ITERATIONS):
    try:
        bids, asks = get_order_book(SYMBOL)
        price = get_price(SYMBOL)

        mid_price, spread, bid_depth, ask_depth = compute_metrics(bids, asks)

        results.append({
            "iteration": i,
            "price": price,
            "mid_price": mid_price,
            "spread": spread,
            "bid_depth": bid_depth,
            "ask_depth": ask_depth
        })

        print(f"Iteration {i+1}: Price={price:.2f}, Spread={spread:.4f}")
        
        time.sleep(SLEEP_TIME)

    except Exception as e:
        print("Error:", e)
        time.sleep(SLEEP_TIME)

df = pd.DataFrame(results)
df.head()

# Rolling statistics
df['price_mean'] = df['price'].rolling(WINDOW_SIZE).mean()
df['price_std'] = df['price'].rolling(WINDOW_SIZE).std()

df['spread_mean'] = df['spread'].rolling(WINDOW_SIZE).mean()
df['spread_std'] = df['spread'].rolling(WINDOW_SIZE).std()

# Z-scores
df['price_z'] = (df['price'] - df['price_mean']) / df['price_std']
df['spread_z'] = (df['spread'] - df['spread_mean']) / df['spread_std']

df['price_anomaly'] = df['price_z'].abs() > Z_THRESHOLD
df['spread_anomaly'] = df['spread_z'].abs() > Z_THRESHOLD

df.tail(10)

# Price
df['price'].plot(title="Price Over Time")
plt.xlabel("Iteration")
plt.ylabel("Price")
plt.show()

# Spread
df['spread'].plot(title="Spread Over Time")
plt.xlabel("Iteration")
plt.ylabel("Spread")
plt.show()

# Price Z-score
df['price_z'].plot(title="Price Z-Score")
plt.axhline(2)
plt.axhline(-2)
plt.show()

# Spread Z-score
df['spread_z'].plot(title="Spread Z-Score")
plt.axhline(2)
plt.axhline(-2)
plt.show()

print("Number of price anomalies:", df['price_anomaly'].sum())
print("Number of spread anomalies:", df['spread_anomaly'].sum())

df[['price', 'spread', 'price_z', 'spread_z']].describe()